# 003 · Generator Expressions and Lazy Evaluation

> **Syntax › Iterators & Generators** · Python 3.14

A **generator expression** is a comprehension in round brackets: `(x * 2 for x in xs)`. It produces values lazily, one at a time — the Python counterpart of a LINQ query with **deferred execution**.

**In this notebook you will learn:**

- the difference between a generator expression and a list comprehension
- why lazy evaluation saves memory (measured with `sys.getsizeof`)
- how deferred execution compares to LINQ
- how to chain lazy pipelines
- a first look at `itertools`: `islice`, `takewhile`, `chain`, `count`, `batched`
- the single-pass gotcha (≈ C#'s "possible multiple enumeration" warning)

## C# mental model

| C# (LINQ) | Python |
|---|---|
| `xs.Select(x => x * 2)` — lazy | `(x * 2 for x in xs)` — lazy |
| `xs.Select(x => x * 2).ToList()` — eager | `[x * 2 for x in xs]` — eager |
| `xs.Where(p).Select(f)` | `(f(x) for x in xs if p(x))` |
| `Take(n)` / `Skip(n)` | `itertools.islice` |
| `TakeWhile` / `SkipWhile` | `itertools.takewhile` / `itertools.dropwhile` |
| `Concat` | `itertools.chain` |
| `Chunk(n)` (.NET 6+) | `itertools.batched(xs, n)` (3.12+) |

The big difference: a LINQ query is a re-runnable *recipe* (an `IEnumerable<T>`), but a generator expression is an **iterator** — single-use.

## Generator expression vs list comprehension

Same syntax, different brackets *(comprehensions are covered in Collections › 007 · Comprehensions)*:

- `[...]` — **list comprehension**: runs immediately, builds the whole list
- `(...)` — **generator expression**: builds nothing yet, yields on demand

In [ ]:
nums = [1, 2, 3, 4, 5]

squares_list = [n * n for n in nums]    # C#: .Select(n => n * n).ToList()
squares_gen = (n * n for n in nums)     # C#: .Select(n => n * n)

print(squares_list)
print(squares_gen)          # <generator object ...> — no values yet
print(list(squares_gen))    # materialise it

When a generator expression is the **only** argument to a function, you can drop the extra parentheses. This is the idiomatic way to feed aggregates:

In [ ]:
orders = [("widget", 3, 2.50), ("gadget", 1, 10.00), ("doohickey", 4, 1.25)]

total = sum(qty * price for _, qty, price in orders)         # no [] needed
has_bulk = any(qty >= 4 for _, qty, _ in orders)
longest = max(len(name) for name, _, _ in orders)

print(total, has_bulk, longest)

With more than one argument, the parentheses are required:

In [ ]:
print(sorted((name for name, _, _ in orders), reverse=True))

## Memory: lazy vs eager

A list comprehension stores every result. A generator expression stores only its current state, so its size is constant no matter how many items it will produce. (`sys.getsizeof` reports the container's own size, not the size of the items it references.)

In [ ]:
import sys

eager = [n * n for n in range(1_000_000)]
lazy = (n * n for n in range(1_000_000))

print(f"list comprehension:   {sys.getsizeof(eager):>10,} bytes")
print(f"generator expression: {sys.getsizeof(lazy):>10,} bytes")
print(sum(lazy) == sum(eager))   # same results

💡 Prefer a generator expression when you consume the values **once** (sum, any, writing to a file, feeding another step). Prefer a list when you need `len()`, indexing, or multiple passes.

## Deferred execution — just like LINQ

```csharp
// C#
var source = new List<int> { 1, 2, 3 };
var query = source.Select(x => x * 10);   // nothing runs yet
source.Add(4);
Console.WriteLine(string.Join(", ", query));   // 10, 20, 30, 40
```

The generator expression captures the *iterable*, and evaluates the expression body only when values are pulled:

In [ ]:
source = [1, 2, 3]
query = (x * 10 for x in source)   # nothing runs yet
source.append(4)
print(list(query))                 # [10, 20, 30, 40] — saw the appended item

⚠️ **Gotcha:** the **outermost** iterable expression is evaluated immediately (when the generator is created); everything else is deferred. Rebinding the variable afterwards does not affect the generator:

In [ ]:
source = [1, 2, 3]
query = (x * 10 for x in source)   # `source` (the list object) captured now
source = [100, 200]                # rebinding the name — query doesn't care
print(list(query))                 # [10, 20, 30]

You can watch the laziness by putting a side effect in the pipeline. Items flow through **one at a time**, not stage by stage — exactly like a LINQ chain:

In [ ]:
def log(stage, x):
    print(f"  {stage}({x})")
    return x


pipeline = (log("select", x * 2) for x in range(5) if log("where", x) % 2 == 0)
print("created")
print("first result:", next(pipeline))
print("second result:", next(pipeline))

## Chaining lazy pipelines

Each generator expression can consume the previous one. Nothing runs until the final consumer pulls values.

```csharp
// C#
var total = lines
    .Select(l => l.Trim())
    .Where(l => l.Length > 0 && !l.StartsWith('#'))
    .Select(int.Parse)
    .Sum();
```

In [ ]:
lines = ["  10", "# comment", "", "20 ", "  5  "]

trimmed = (line.strip() for line in lines)
meaningful = (line for line in trimmed if line and not line.startswith("#"))
values = (int(line) for line in meaningful)

print(sum(values))   # 35 — the whole pipeline runs here, once

## `itertools` teaser

The `itertools` module is Python's toolbox of lazy iterator building blocks — think of it as a big part of `System.Linq`. This is only a preview; it gets fuller coverage later in the Standard Library group.

| C# (LINQ) | `itertools` |
|---|---|
| `Take(n)`, `Skip(n)` | `islice(xs, n)`, `islice(xs, n, None)` |
| `TakeWhile(p)` | `takewhile(p, xs)` |
| `SkipWhile(p)` | `dropwhile(p, xs)` |
| `Concat(other)` | `chain(xs, other)` |
| `Enumerable.Range(start, int.MaxValue)` | `count(start)` (infinite) |
| `Chunk(n)` | `batched(xs, n)` (3.12+) |

In [ ]:
from itertools import batched, chain, count, dropwhile, islice, takewhile

print(list(islice(count(1), 5)))                     # Take(5) of an infinite counter
print(list(islice("abcdefgh", 2, 6)))                # Skip(2).Take(4)
print(list(takewhile(lambda n: n < 4, [1, 2, 3, 7, 1])))   # TakeWhile -> [1, 2, 3]
print(list(dropwhile(lambda n: n < 4, [1, 2, 3, 7, 1])))   # SkipWhile -> [7, 1]
print(list(chain([1, 2], (3, 4), range(5, 7))))      # Concat

`batched()` yields **tuples** (C#'s `Chunk` yields arrays). The last batch may be shorter; pass `strict=True` (3.13+) to raise instead.

In [ ]:
for batch in batched(range(7), 3):
    print(batch)

try:
    list(batched(range(7), 3, strict=True))
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

Combining an infinite source with lazy filters and `islice` is a common pattern — "the first 5 even squares":

In [ ]:
even_squares = (n * n for n in count(1) if n % 2 == 0)
print(list(islice(even_squares, 5)))

## ⚠️ Gotcha: single pass (≈ "possible multiple enumeration")

In C#, enumerating an `IEnumerable<T>` twice re-runs the query (and ReSharper/Rider warn about *possible multiple enumeration*). In Python, enumerating a generator twice gives you **nothing** the second time — silently.

In [ ]:
scores = (s for s in [70, 85, 90])

print(max(scores))   # 90
print(sum(scores))   # 0 — already exhausted, no error!

This is especially sneaky when a function receives "some iterable" and loops over it twice:

In [ ]:
def average(values):
    return sum(values) / len(list(values))   # second pass over `values`!


try:
    print(average(x for x in [1, 2, 3]))
except ZeroDivisionError as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
def average_safe(values):
    items = list(values)     # materialise once, like .ToList()
    return sum(items) / len(items)


print(average_safe(x for x in [1, 2, 3]))

💡 If you need a reusable lazy "query", wrap it in a function (or a class with `__iter__`) and call it each time — that's the closest equivalent of a re-enumerable LINQ `IEnumerable<T>`.

In [ ]:
data = [3, 1, 4, 1, 5]


def big_values():
    return (x for x in data if x > 2)   # fresh generator on every call


print(list(big_values()), list(big_values()))

## Summary

| Task | C# | Python |
|---|---|---|
| Lazy projection | `xs.Select(f)` | `(f(x) for x in xs)` |
| Eager projection | `xs.Select(f).ToList()` | `[f(x) for x in xs]` |
| Filter + project | `xs.Where(p).Select(f)` | `(f(x) for x in xs if p(x))` |
| Aggregate | `xs.Sum(x => x.Qty)` | `sum(x.qty for x in xs)` |
| Take / Skip | `Take(n)` / `Skip(n)` | `islice(xs, n)` / `islice(xs, n, None)` |
| TakeWhile | `TakeWhile(p)` | `takewhile(p, xs)` |
| Concat | `a.Concat(b)` | `chain(a, b)` |
| Chunk | `Chunk(n)` | `batched(xs, n)` |
| Re-enumerate | re-runs the query | exhausted — rebuild or `list()` first |

## Exercises

1. Using a generator expression, compute the total length of all words in `"lazy evaluation is great"` without building a list.
2. Using `count`, a generator expression and `islice`, produce the first 6 numbers divisible by both 3 and 4.
3. The code below prints `True` and then `False`. Explain why, and fix it so both checks work.
   ```python
   gen = (n for n in range(10))
   print(5 in gen)
   print(3 in gen)
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
from itertools import count, islice

# 1
print(sum(len(w) for w in "lazy evaluation is great".split()))   # 21

# 2
print(list(islice((n for n in count(1) if n % 12 == 0), 6)))
# [12, 24, 36, 48, 60, 72]

# 3 — `in` consumes the generator up to and including 5, so 3 is already gone.
#     Use a reusable collection (or a range/set) instead:
nums = range(10)        # or list(...) / set(...)
print(5 in nums, 3 in nums)   # True True
```

</details>